### Comparação entre Métodos de Similaridade de Texto (Fuzzy Matching)

| Método                 | O que faz?                                                                 | Ordem das palavras importa? | Ideal para                                             | Fragilidade principal                              |
|------------------------|---------------------------------------------------------------------------|------------------------------|--------------------------------------------------------|----------------------------------------------------|
| `fuzz.ratio`           | Compara diretamente os textos usando distância de Levenshtein             | ✅ Sim                       | Frases curtas e exatas                                 | Penaliza mudanças simples na ordem das palavras    |
| `fuzz.token_sort_ratio`| Ordena as palavras alfabeticamente antes de comparar                      | ❌ Não                      | Frases com as mesmas palavras, mas em ordem diferente  | Palavras irrelevantes ainda impactam a pontuação   |
| `fuzz.partial_ratio`   | Compara a melhor substring entre os textos (menor dentro do maior)         | ❓ Parcialmente              | Quando a frase é parte de outra mais longa             | Pode gerar falsos positivos por similaridade parcial|


In [1]:
# Manipulação e Tratamento de dados
import openpyxl
import pandas as pd
import numpy as np2 
from numpy import NaN

#ignorando Warning inuteis
import warnings 
from pandas.errors import SettingWithCopyWarning
warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)
warnings.filterwarnings(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

#definindo quantidade de linhas e colunas visiveis
pd.set_option('display.max_rows', 25)
pd.set_option('display.max_columns', 500)

# Repositorio - Diretorio
import shutil
import os

In [3]:
pasta = 'H:/Drives compartilhados/Qualidade/NOTIFICAÇÕES E ANÁLISE DE EVENTOS/NOTIFICAÇÕES 2025/04 - Abril/'
conteudo = os.listdir(pasta)

df = pd.DataFrame()
for arquivo in conteudo:
    fd = pd.read_excel(pasta + arquivo, sheet_name='Notificações')
    df = pd.concat([df, fd], axis=0)

df = df.reset_index(drop=True)
df.sample(2)
print(df.shape)

#################################################################################

pasta_segmentacao = 'H:/Drives compartilhados/Qualidade/NOTIFICAÇÕES E ANÁLISE DE EVENTOS/NOTIFICAÇÕES 2025/0 - Segmentação dos Dados/'
fd = pd.read_excel(pasta_segmentacao+'qual_incidente_ocorreu.xlsx')
fd.sample(2)
print(fd.shape)

(267, 20)
(312, 2)


In [4]:
print(len(df))
print(len(df['Qual incidente ocorreu'].dropna().astype(str).unique()))

count = 0
count_erro = 0

for i in sorted(df['Qual incidente ocorreu'].dropna().astype(str).unique()):
    if i in fd['lista'].to_list():
        count += 1
        #print(i)
    else:
        count_erro += 1
        print('"'+i+'",')

print(count)
print(count_erro)

267
132
"Agendamento inadequado de procedimento",
"Atraso em cirurgia robótica",
"Atraso médico",
"Atraso na agenda",
"Atraso na assistência",
"Atraso na reserva de sangue",
"Atraso no fluxo cirúrgico",
"Atraso no início de cirurgia",
"Atraso por falha de logística",
"Atraso por falha de processo",
"Equipamento ausente",
"Equipamento com mau funcionamento",
"Equipamento danificado",
"Equipamento desatualizado",
"Equipamento inadequado para o paciente",
"Equipamento inadequado para posicionamento",
"Equipamento não carregado",
"Erro de dispensação de medicação",
"Erro de registro em sistema",
"Falha de anestésico",
"Falha de bloqueio anestésico",
"Falha de infraestrutura",
"Falha de preparo cirúrgico",
"Falha de protocolo de acesso venoso em obstetrícia",
"Falha de registro no sistema",
"Falha em equipamento de anestesia",
"Falha em equipamento de transporte",
"Falha em equipamento médico",
"Falha na administração de medicamentos",
"Falha na administração de medicação pré-anestésica (MP

In [5]:
from rapidfuzz import process, fuzz

# Preparar listas únicas
lista_padrao = fd['lista'].dropna().astype(str).unique().tolist()
lista_nova = df['Qual incidente ocorreu'].dropna().astype(str).unique().tolist()

# Parâmetros
LIMIAR_SIMILARIDADE = 85  # porcentagem mínima para considerar uma correspondência

# Armazenar resultados
correspondencias = []
nao_correspondidos = []

# Loop de correspondência
for texto in sorted(lista_nova):
    melhor_correspondencia, score, _ = process.extractOne(
        query=texto,
        choices=lista_padrao,
        scorer=fuzz.token_sort_ratio  # ou fuzz.ratio, fuzz.partial_ratio...
    )
    
    if score >= LIMIAR_SIMILARIDADE:
        correspondencias.append({
            'texto_origem': texto,
            'texto_encontrado': melhor_correspondencia,
            'similaridade': score
        })
    else:
        nao_correspondidos.append({
            'texto_origem': texto,
            'melhor_tentativa': melhor_correspondencia,
            'similaridade': score
        })

# Converter em DataFrames
df_correspondencias = pd.DataFrame(correspondencias)
df_nao_correspondidos = pd.DataFrame(nao_correspondidos)

print(len(df_correspondencias))
display(df_correspondencias.sample(1))

print(len(df_nao_correspondidos))
display(df_nao_correspondidos.sort_values(by='similaridade'))

50


,texto_origem,texto_encontrado,similaridade
30,Falha na orientação/educação do paciente ou fa...,Falha na orientação/educação do paciente ou fa...,100.0


82


,texto_origem,melhor_tentativa,similaridade
78,Quebra de fluxo em pacientes usuários de análo...,Contrafluxo de pacientes para unidade de maior...,50.000000
76,Problema em monitorização pós-anestésica,PCR em unidade de internação,50.000000
61,Falta de infraestrutura adequada,Falha na prescrição da dieta,53.333333
74,Intercorrência intraoperatória,Abertura involuntária da ferida operatória (de...,54.117647
11,Equipamento com mau funcionamento,Equipamento/infraestrutura inadequado,54.285714
...,...,...,...
30,Falha na assistência durante o exame,Falha durante a assistência à saúde,81.690141
31,Falha na comunicação entre os setores,Falha/ausência de comunicação entre os setores,81.927711
39,Falha na monitorização do paciente,Falha na identificação do paciente,82.352941
5,Atraso na reserva de sangue,Falha na reserva de sangue,83.018868


In [7]:
from rapidfuzz import process, fuzz

# Exemplo de entrada
# query = df_nao_correspondidos['texto_origem'][df_nao_correspondidos['texto_origem'].sample(1).index[0]]
query = df_nao_correspondidos['texto_origem'][1]
lista_padrao = fd['lista'].dropna().astype(str).unique().tolist()
print('frase procuradas:', query, '\n'+'Correspondencias:', '*'*75)

# Usando token_sort_ratio
print("\n=== token_sort_ratio ===")
resultados_token_sort = process.extract(query, lista_padrao, scorer=fuzz.token_sort_ratio, limit=7)
for i, (texto, score, _) in enumerate(resultados_token_sort, 1):
    print(f"{i}º: {texto} ({round(score,1)})")

# Usando ratio
print("\n=== ratio ===")
resultados_ratio = process.extract(query, lista_padrao, scorer=fuzz.ratio, limit=7)
for i, (texto, score, _) in enumerate(resultados_ratio, 1):
    print(f"{i}º: {texto} ({round(score,1)})")

# Usando partial_ratio
print("\n=== partial_ratio ===")
resultados_partial = process.extract(query, lista_padrao, scorer=fuzz.partial_ratio, limit=7)
for i, (texto, score, _) in enumerate(resultados_partial, 1):
    print(f"{i}º: {texto} ({round(score,1)})")


frase procuradas: Atraso em cirurgia robótica 
Correspondencias: ***************************************************************************

=== token_sort_ratio ===
1º: Atraso na abertura de ficha (63.0)
2º: Atraso do procedimento cirúrgico (61.0)
3º: Atraso na entrega de exames (55.6)
4º: Lesão de orgão durante cirurgia (55.2)
5º: Cirurgia em órgão errado (54.9)
6º: Atraso na entrega medicamento (53.6)
7º: Atraso na limpeza do leito/quarto/sala cirúrgica (53.3)

=== ratio ===
1º: Atraso na abertura de ficha (59.3)
2º: Atraso do procedimento cirúrgico (57.6)
3º: Reabordagem cirúrgica (54.2)
4º: Atraso na entrega medicamento (53.6)
5º: Atraso na entrega da dieta (52.8)
6º: Reabordagem cirúrgica não programada (50.8)
7º: Atraso na entrega de exames (48.1)

=== partial_ratio ===
1º: Reabordagem cirúrgica (61.5)
2º: Outro (60.0)
3º: Atraso na abertura de ficha (59.3)
4º: Exposição repetida de órgãos pela ferida operatória após a cirurgia (evisceração) (59.3)
5º: Reabordagem cirúrgica não

### Quando usar cada método?

| Situação do dado                                      | Método recomendado         |
|--------------------------------------------------------|----------------------------|
| Frases curtas, exatas e bem definidas                 | `fuzz.ratio`              |
| Mesmas palavras, mas fora de ordem                    | `fuzz.token_sort_ratio`   |
| A frase analisada é parte de uma frase maior          | `fuzz.partial_ratio`      |
| Comparar todas as abordagens e pegar a melhor similaridade | Usar todos e comparar     |
